In [6]:
import Pkg;
Pkg.activate(@__DIR__);
Pkg.status()
Pkg.instantiate()

Status `~/Desktop/Evolutionary-Algos/rdex/src/Project.toml`
  [09f84164] HypothesisTests v0.11.6


  Activating project at `~/Desktop/Evolutionary-Algos/rdex/src`


In [7]:
using LinearAlgebra
using DelimitedFiles
using Statistics
using HypothesisTests
using Printf

In [8]:
function load_final_values(folder, func)
    ffile = joinpath(folder, @sprintf("RDEx_f_F%d_D30_.txt", func))
    cfile = joinpath(folder, @sprintf("RDEx_C_F%d_D30.txt", func))

    fdata = readdlm(ffile)
    cdata = readdlm(cfile)

    final_ev = fdata[:, end]
    final_cv = sum(abs.(cdata[:, end-8:end]), dims=2)[:]

    return final_ev, final_cv
end;

In [9]:
algorithms = Dict(
    "Random" => "NoMods",
    "Sobol"  => "SobolInit",
    "Halton" => "HaltonInit",
    "ExtArch" => "ExtArchive",
    "ExtArch2" => "ExtArchive2"
)

num_funcs = 28
num_runs = 25

results = Dict()

for (name, path) in algorithms
    alg_res = Dict()

    for f in 1:num_funcs
        ev, cv = load_final_values(path, f)

        alg_res[f] = (ev=ev, cv=cv)
    end

    results[name] = alg_res
end

In [11]:
println("----------------------------------------------------------------------------------------------------------------------")
println("Function | Random(mean±std)   | Sobol(mean±std)    |  Halton(mean±std)  |  ExtArch(mean±std)  |  ExtArch2(mean±std)")
println("----------------------------------------------------------------------------------------------------------------------")

for f in 1:num_funcs
    stats = Dict()

    for alg in keys(algorithms)
        ev = results[alg][f].ev
        stats[alg] = (mean(ev), std(ev))
    end

    @printf(
        "F%-2d\t | %.3e±%.2e | %.3e±%.2e | %.3e±%.2e | %.3e±%.2e | %.3e±%.2e\n",
        f,
        stats["Random"][1], stats["Random"][2],
        stats["Sobol"][1],  stats["Sobol"][2],
        stats["Halton"][1], stats["Halton"][2],
        stats["ExtArch"][1], stats["ExtArch"][2],
        stats["ExtArch2"][1], stats["ExtArch2"][2]
    )
end

println("\n-------------------------------------------------------")
println("Wilcoxon Signed Rank Test (Random vs Others)")
println("-------------------------------------------------------")

wins = Dict("Sobol"=>0,"Halton"=>0,"ExtArch"=>0, "ExtArch2"=>0)
loss = Dict("Sobol"=>0,"Halton"=>0,"ExtArch"=>0, "ExtArch2"=>0)
ties = Dict("Sobol"=>0,"Halton"=>0,"ExtArch"=>0, "ExtArch2"=>0)

for f in 1:num_funcs
    base = results["Random"][f].ev

    for alg in ["Sobol","Halton","ExtArch", "ExtArch2"]
        comp = results[alg][f].ev

        test = SignedRankTest(base, comp)

        if pvalue(test) < 0.05
            if mean(comp) < mean(base)
                wins[alg]+=1
            else
                loss[alg]+=1
            end
        else
            ties[alg]+=1
        end
    end
end


println("\nRandom vs Sobol  : ",
        wins["Sobol"], "/", ties["Sobol"], "/", loss["Sobol"])

println("Random vs Halton : ",
        wins["Halton"], "/", ties["Halton"], "/", loss["Halton"])

println("Random vs ExtArch : ", 
        wins["ExtArch"], "/", ties["ExtArch"], "/", loss["ExtArch"])

println("Random vs ExtArch2 : ", 
        wins["ExtArch2"], "/", ties["ExtArch2"], "/", loss["ExtArch2"])

---------------------------------------------------------------------------------------------------
Function | Random(mean±std)   | Sobol(mean±std)    |  Halton(mean±std)  |  ExtArch(mean±std)
---------------------------------------------------------------------------------------------------
F1 	 | 1.055e+05±1.94e+03 | 1.175e+05±2.26e+03 | 1.059e+05±1.77e+03 | 1.169e+05±2.35e+03
F2 	 | 1.081e+05±2.26e+03 | 1.185e+05±2.36e+03 | 1.095e+05±2.05e+03 | 1.209e+05±2.23e+03
F3 	 | 6.000e+05±0.00e+00 | 6.000e+05±0.00e+00 | 6.000e+05±0.00e+00 | 6.000e+05±0.00e+00
F4 	 | 6.000e+05±0.00e+00 | 6.000e+05±0.00e+00 | 6.000e+05±0.00e+00 | 6.000e+05±0.00e+00
F5 	 | 1.849e+05±3.22e+03 | 1.939e+05±2.52e+03 | 1.855e+05±3.26e+03 | 1.913e+05±2.24e+03
F6 	 | 1.841e+05±1.57e+05 | 2.298e+05±1.89e+05 | 1.847e+05±1.57e+05 | 2.326e+05±1.87e+05
F7 	 | 2.014e+05±4.03e+04 | 2.828e+05±7.68e+04 | 1.922e+05±4.46e+04 | 2.249e+05±9.64e+04
F8 	 | 1.226e+05±1.98e+03 | 1.308e+05±1.30e+03 | 1.214e+05±1.91e+03 | 1.331e+05±2.53